In [ ]:
# 2. Configurer le dossier Kaggle en arrière-plan
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Télécharger le dataset (le nom correspond à la fin de votre URL Kaggle)
!kaggle datasets download -d ericanacletoribeiro/cicids2017-cleaned-and-preprocessed

# 4. Dézipper le fichier téléchargé
import zipfile
import os

zip_file = "cicids2017-cleaned-and-preprocessed.zip"
if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall("cicids2017_data")
    print("Le dataset a été téléchargé et dézippé avec succès !")
    # Afficher les fichiers extraits pour voir le nom exact du CSV
    print("Fichiers disponibles :", os.listdir("cicids2017_data"))
else:
    print("Erreur lors du téléchargement.")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/ericanacletoribeiro/cicids2017-cleaned-and-preprocessed
License(s): CC0-1.0
100% 200M/200M [00:01<00:00, 138MB/s]

Le dataset a été téléchargé et dézippé avec succès !
Fichiers disponibles : ['cicids2017_cleaned.csv']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# 1. Définir le chemin exact du fichier (visible sur votre image)
chemin_csv = "cicids2017_data/cicids2017_cleaned.csv"

print("Chargement du dataset en cours (cela peut prendre quelques secondes)...")

df = pd.read_csv(chemin_csv)

print("Dataset chargé avec succès ! ")

# 3. Afficher les dimensions du dataset (Lignes, Colonnes)
print(f"Dimensions : {df.shape[0]} lignes et {df.shape[1]} colonnes.\n")
df.head()

Chargement du dataset en cours (cela peut prendre quelques secondes)...
Dataset chargé avec succès ! 
Dimensions : 2520751 lignes et 53 colonnes.



,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [ ]:
df['Attack Type'].value_counts()

,count
Attack Type,
Normal Traffic,2095057
DoS,193745
DDoS,128014
Port Scanning,90694
Brute Force,9150
Web Attacks,2143
Bots,1948


In [ ]:
chemin_drive = "/content/drive/MyDrive/cicids2017_final.csv"

df.to_csv(chemin_drive, index=False)
print(" Fichier sauvegardé définitivement sur  Google Drive !")

 Fichier sauvegardé définitivement sur  Google Drive !


In [ ]:
# Run this in Colab to download the file directly to your PC
from google.colab import files
files.download("/content/drive/MyDrive/cicids2017_final.csv")

# **PROCESS NOW START FROM HERE**

In [ ]:
# 1. Define the exact whitelist of the top 20 most important columns + the Label
important_columns = [
    'Destination Port',
    'Init_Win_bytes_forward',
    'Init_Win_bytes_backward',
    'Flow Duration',
    'Total Length of Fwd Packets',
    'Total Length of Bwd Packets',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Bwd Packet Length Mean',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Average Packet Size',
    'Flow IAT Max',
    'Flow IAT Mean',
    'Fwd IAT Max',
    'Bwd Header Length',
    'Fwd Header Length',
    'PSH Flag Count',
    'ACK Flag Count',
    'Attack Type'
]

# 2. Filter the DataFrame to keep ONLY these columns
# We use errors='ignore' in case some names have slightly different spacing in your CSV
df_optimized = df[[col for col in important_columns if col in df.columns]]

print(f"Original shape: {df.shape}")
print(f"Optimized shape: {df_optimized.shape}")
print("\nYour dataset is now lightweight, hyper-focused, and ready for modeling!")

In [ ]:
df_optimized.head()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

import time
import psutil
import threading

import joblib

In [ ]:
def apply_rf(X_train, y_train, best_params=None, random_state=42, n_jobs=-1, cv=5):
    """
    Entraîne un RandomForest et mesure temps d'entraînement + mémoire.

    Returns:
        cv_scores_rf, measurement_rf (dict), rf_model (entraîné)
    """
    best_params = best_params or {}
    rf_model = RandomForestClassifier(**best_params, random_state=random_state, n_jobs=n_jobs)
    rf_model.fit(X_train, y_train)


    cv_scores_rf = cross_val_score(rf_model, X_train, y_train, cv=cv, n_jobs=n_jobs)



    return cv_scores_rf , rf_model

In [ ]:
from sklearn.preprocessing import LabelEncoder
def apply_xgboost(X_train, y_train, best_params=None, random_state=42, n_jobs=-1, cv=5):
    """
    Entraîne un XGBoost (labels encodés) et mesure temps d'entraînement + mémoire.

    Returns:
        cv_scores_xgb, measurement_xgb (dict), xgb_model (entraîné), label_encoder
    """
    best_params = best_params or {}

    # XGBoost multi:softmax exige des labels numériques 0..N-1
    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train)

    xgb_model = xgb.XGBClassifier(
        **best_params,
        objective='multi:softmax',
        num_class=len(label_encoder.classes_),
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric='mlogloss'
    )



    xgb_model.fit(X_train, y_train_enc)


    cv_scores_xgb = cross_val_score(xgb_model, X_train, y_train_enc, cv=cv, n_jobs=n_jobs)


    return cv_scores_xgb , xgb_model, label_encoder

In [ ]:
def apply_knn(X_train, y_train, best_params=None, n_jobs=-1, cv=5):
    """
    Entraîne un KNN et mesure temps d'entraînement + mémoire.

    Returns:
        cv_scores_knn, measurement_knn (dict), knn_model (entraîné)
    """
    best_params = best_params or {}
    knn_model = KNeighborsClassifier(**best_params, n_jobs=n_jobs)



    knn_model.fit(X_train, y_train)


    cv_scores_knn = cross_val_score(knn_model, X_train, y_train, cv=cv, n_jobs=n_jobs)



    return cv_scores_knn, knn_model

In [ ]:
X = clean_df.drop('Attack Type', axis=1)
y = clean_df['Attack Type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:

from imblearn.under_sampling import RandomUnderSampler

target_counts = {
    'Normal Traffic': 20000,
    'DoS': 3000,
    'DDoS': 1500,
    'Port Scanning': 1000,
    'Brute Force': 700,
    'Web Attacks': 160,
    'Bots': 140
}


X_train_resampled, y_train_resampled = RandomUnderSampler(
    sampling_strategy=target_counts, random_state=42
).fit_resample(X_train, y_train)

print(y_train_resampled.value_counts())
print("Total:", len(y_train_resampled))

# Scaling après le resampling (fit uniquement sur les données déjà resamplées)
X_train_resampled_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

# On garde X_test / y_test : ils servent plus bas pour l'évaluation.
# On NE supprime PAS clean_df, car il est encore utile pour d'éventuelles vérifications.
del X_train, y_train, X, y

In [ ]:
# --- Random Forest (entraîné sur les données non scalées, sous-échantillonnées) ---
best_params_rf = {
    'n_estimators': 100,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'max_depth': None
}

cv_sc_rf , rf_model = apply_rf(
    X_train_resampled, y_train_resampled, best_params=best_params_rf
)

print("RF - CV scores :", cv_sc_rf, "| moyenne :", np.mean(cv_sc_rf))



In [ ]:
# Random Forest
y_pred_rf = rf_model.predict(X_test)
print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# --- XGBoost (entraîné sur les données scalées + sur-échantillonnées) ---
best_params_xgb = {
    'n_estimators': 100,
    'max_depth': 8,
    'learning_rate': 0.1
}

cv_sc_xgb , xgb_model, xgb_label_encoder = apply_xgboost(
    X_train_resampled_scaled, y_train_resampled, best_params=best_params_xgb
)

print("XGB - CV scores :", cv_sc_xgb, "| moyenne :", np.mean(cv_sc_xgb))


# XGBoost
y_pred_xgb_enc = xgb_model.predict(X_test_scaled)
y_pred_xgb = xgb_label_encoder.inverse_transform(y_pred_xgb_enc)
print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# --- KNN (entraîné sur les données scalées + sur-échantillonnées) ---
best_params_knn = {
    'n_neighbors': 5
}

cv_sc_knn, knn_model = apply_knn(
    X_train_resampled_scaled, y_train_resampled, best_params=best_params_knn
)

print("KNN - CV scores :", cv_sc_knn, "| moyenne :", np.mean(cv_sc_knn))


# KNN
y_pred_knn = knn_model.predict(X_test_scaled)
print("=== KNN ===")
print(classification_report(y_test, y_pred_knn))

In [ ]:
# Sauvegarde des modèles entraînés
joblib.dump(rf_model, 'rf_model.joblib')
joblib.dump(xgb_model, 'xgb_model.joblib ')
joblib.dump(xgb_label_encoder, 'xgb_label_encoder.joblib')
joblib.dump(knn_model, 'knn_model.joblib')
print("Modèles sauvegardés (rf_model.joblib, xgb_model.joblib, xgb_label_encoder.joblib, knn_model.joblib)")